# Evaluate Starlar Fine-Tuned LLM on Old Best RAG Contexts

This notebook evaluates the Starlar v2 fine-tuned Mistral QLoRA model using the exact retrieved contexts from the previous best RAG pipeline.

Goal:
- Keep retrieval fixed.
- Use the old best retrieved contexts.
- Replace only the generator with the Starlar fine-tuned LLM.
- Compare old best base-generator answers with Starlar fine-tuned generator answers.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > L4 GPU seç.")

CUDA available: True
GPU: NVIDIA L4


In [3]:
import os

project_path = "/content/drive/MyDrive/turkish_legal_rag"

outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

old_best_scored_path = f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_scored.csv"

adapter_path = f"{models_path}/mistral_legal_qlora_starlar_v2_800steps"

print("Old best scored exists:", os.path.exists(old_best_scored_path))
print("Adapter folder exists:", os.path.exists(adapter_path))
print("Adapter config exists:", os.path.exists(f'{adapter_path}/adapter_config.json'))
print("Adapter model exists:", os.path.exists(f'{adapter_path}/adapter_model.safetensors'))

Old best scored exists: True
Adapter folder exists: True
Adapter config exists: True
Adapter model exists: True


In [4]:
!pip install -q -U transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 156.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.7 MB/s eta 0:00:00


In [5]:
import os
import gc
import json
import ast
import re
import difflib
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

In [6]:
old_best_df = pd.read_csv(old_best_scored_path)

print("Old best shape:", old_best_df.shape)
print("Columns:")
print(old_best_df.columns.tolist())

display(old_best_df[[
    "question",
    "expected_answer",
    "clean_generated_answer",
    "manual_score",
    "is_valid_sample",
    "top1_chunk_id",
    "top1_source",
    "retrieved_contexts"
]].head())

Old best shape: (20, 16)
Columns:
['question', 'expected_answer', 'generated_answer', 'clean_generated_answer', 'top1_chunk_id', 'top1_source', 'top1_source_filter', 'top1_context', 'top1_article_bonus', 'top1_original_rank', 'top1_rerank_rank', 'top1_rerank_score', 'top1_fusion_score', 'retrieved_contexts', 'is_valid_sample', 'manual_score']


,question,expected_answer,clean_generated_answer,manual_score,is_valid_sample,top1_chunk_id,top1_source,retrieved_contexts
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",0.0,True,chunk_000537,Türkiye Cumhuriyeti Anayasası,Cumhurbaşkanlığı seçiminde birinci oylamada ge...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...",0.5,True,chunk_000274,Türkiye Cumhuriyeti Anayasası,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",0.5,True,chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 13/5/1981 gün eklendi.,0.0,True,chunk_003745,Türkiye Cumhuriyeti İş Kanunu,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde hakkının kullanılmasının e...",0.5,True,chunk_003431,Türk Ceza Kanunu,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI B...


In [7]:
def parse_contexts(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value)

    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
    except:
        pass

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return parsed
    except:
        pass

    return [text]


eval_df = old_best_df.copy()
eval_df["parsed_contexts"] = eval_df["retrieved_contexts"].apply(parse_contexts)

print("Context count distribution:")
display(eval_df["parsed_contexts"].apply(len).value_counts())

display(eval_df[[
    "question",
    "expected_answer",
    "parsed_contexts",
    "manual_score"
]].head())

Context count distribution:


,count
parsed_contexts,
1,20


,question,expected_answer,parsed_contexts,manual_score
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,[Cumhurbaşkanlığı seçiminde birinci oylamada g...,0.0
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","[Madde 10 – Herkes, dil, ırk, renk, cinsiyet, ...",0.5
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","[Madde 17 – Herkes, yaşama, maddi ve manevi va...",0.5
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,[31/7/2008-5797/10 md.) Bu fıkrada düzenlenen ...,0.0
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,[ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI ...,0.5


In [8]:
def truncate_text(text, max_chars=1400):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].strip()


def build_context_block(contexts, max_contexts=3, max_chars_per_context=1400):
    selected_contexts = contexts[:max_contexts]

    blocks = []

    for i, ctx in enumerate(selected_contexts, start=1):
        blocks.append(
            f"[Bağlam {i}]\n{truncate_text(ctx, max_chars=max_chars_per_context)}"
        )

    return "\n\n".join(blocks)

In [9]:
SYSTEM_INSTRUCTION = """Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma."""


def build_starlar_style_rag_prompt(row):
    question = str(row["question"])
    contexts = row["parsed_contexts"]

    top1_source = str(row.get("top1_source", "Retrieved Legal Context"))
    top1_chunk_id = str(row.get("top1_chunk_id", "unknown_chunk"))

    context_block = build_context_block(
        contexts,
        max_contexts=3,
        max_chars_per_context=1400
    )

    user_content = f"""[Kaynak]
Başlık: Retrieved Legal Contexts
Kaynak: {top1_source}
Chunk ID: {top1_chunk_id}
Citation: {top1_source} - {top1_chunk_id}
Metin:
{context_block}

Soru: {question}"""

    prompt = f"""<s>[INST] {SYSTEM_INSTRUCTION}

{user_content} [/INST]"""

    return prompt


eval_df["starlar_rag_prompt"] = eval_df.apply(build_starlar_style_rag_prompt, axis=1)

print(eval_df.loc[0, "starlar_rag_prompt"][-2000:])

<s>[INST] Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma.

[Kaynak]
Başlık: Retrieved Legal Contexts
Kaynak: Türkiye Cumhuriyeti Anayasası
Chunk ID: chunk_000537
Citation: Türkiye Cumhuriyeti Anayasası - chunk_000537
Metin:
[Bağlam 1]
Cumhurbaşkanlığı seçiminde birinci oylamada gerekli çoğunluğun sağlanamaması halinde 101 inci maddedeki usule göre ikinci oylama yapılır. D. Seçimlerin geriye bırakılması ve ara seçimler [34]

Madde 158 – Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir. [101]

Madde 176 – Anayasanın dayandığı temel görüş ve ilkeleri belirten başlangıç kısmı, Anayasa metnine dahildir.

Soru: Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir? [/INST]


In [10]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Tokenizer ready.")
print("Padding side:", tokenizer.padding_side)
print("Truncation side:", tokenizer.truncation_side)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer ready.
Padding side: right
Truncation side: left


In [11]:
MAX_INPUT_LENGTH = 2048

def generate_answer_only(model, tokenizer, prompt, max_new_tokens=260, max_length=MAX_INPUT_LENGTH):
    tokenizer.truncation_side = "left"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()


def clean_answer(text):
    text = str(text).strip()

    unwanted_markers = [
        "Cevap:",
        "Yanıt:",
        "Answer:"
    ]

    for marker in unwanted_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    return text

In [12]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    common = {}

    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1

    overlap = 0

    for t in ref_tokens:
        if common.get(t, 0) > 0:
            overlap += 1
            common[t] -= 1

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


def text_similarity(prediction, reference):
    return difflib.SequenceMatcher(
        None,
        normalize_text(prediction),
        normalize_text(reference)
    ).ratio()


def has_source_citation(answer):
    answer = str(answer).lower()
    return int("kaynak:" in answer or "citation:" in answer)

In [13]:
ft_base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

ft_base_model.eval()

ft_model = PeftModel.from_pretrained(
    ft_base_model,
    adapter_path
)

ft_model.eval()

print("Starlar fine-tuned Mistral loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Starlar fine-tuned Mistral loaded.


In [14]:
starlar_ft_answers = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    answer = generate_answer_only(
        model=ft_model,
        tokenizer=tokenizer,
        prompt=row["starlar_rag_prompt"],
        max_new_tokens=260,
        max_length=MAX_INPUT_LENGTH
    )

    starlar_ft_answers.append(clean_answer(answer))

eval_df["starlar_finetuned_rag_answer"] = starlar_ft_answers

display(eval_df[[
    "question",
    "expected_answer",
    "clean_generated_answer",
    "starlar_finetuned_rag_answer",
    "manual_score"
]])

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [07:46<00:00, 23.33s/it]


,question,expected_answer,clean_generated_answer,starlar_finetuned_rag_answer,manual_score
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",Tartışmalar:\n\n* Cumhurbaşkanlığı seçiminde b...,0.0
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","[Bağlam 1] Madde 10 – Herkes, dil, ırk, renk, ...",0.5
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",Kaynak metindeki açıklama şu kaynak bilgilerin...,0.5
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 13/5/1981 gün eklendi.,"Geçici madde 20, 13/5/1981 gün ve 2461 sayılı ...",0.0
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde hakkının kullanılmasının e...",Kaynak metindeki esas içerik şudur: [Bağlam 1]...,0.5
5,Cumhurbaşkanının yemin etme zorunluluğu nedir,"Cumhurbaşkanının yemin etme zorunluluğu, anaya...","Cumhurbaşkanın yemin etme zorunluluğu, Cumhurb...",Cumhurbaşkanının yemin etme zorunluluğu:\n[Bağ...,0.0
6,Anayasanın 122. Maddesi Nedir?,"Anayasanın 122. maddesi, sıkıyönetim ilanı ve ...",Anayasanın 122. Maddesi yoktur.,Anayasanın 122. Maddesi: e) Anayasanın halkoyl...,0.0
7,Yasama dokunulmazlığının kaldırılmasına karşı ...,Yasama dokunulmazlığının kaldırılmasına veya m...,"Verilen bağlamlara göre, itiraz süresi yedi gü...",Yasama dokunulmazlığının kaldırılmasına ilişki...,1.0
8,"Bir siyasi parti, çalışma şartlarının belirli ...","Evet, Anayasanın 50. Maddesi, herkesin çalışma...","Evet, Anayasanın 50. Maddesi, siyasi partileri...","[Bağlam 1] Madde 50 – Kimse, yaşına, cinsiyeti...",0.5
9,"Devlet Denetleme Kurulu, Silahlı Kuvvetler üze...","Anayasanın 108. Maddesi, Silahlı Kuvvetlerin D...","Anayasanın 108. Maddesi, Devlet Denetleme Kuru...",108. Maddesi şu içerik ile karşılaşmaktadır: [...,0.0


In [15]:
DIRECT_SYSTEM_INSTRUCTION = """Sen Türk hukuku RAG asistanısın.
Yalnızca verilen bağlamlara dayanarak cevap ver.
Kaynakta olmayan bilgiyi uydurma.
Bu değerlendirmede kaynak/citation satırı yazma.
Bağlam numarası veya chunk bilgisini yazma.
Bağlamı kopyalama.
Sadece sorunun kısa ve doğrudan nihai cevabını Türkçe yaz."""


def build_direct_answer_rag_prompt(row):
    question = str(row["question"])
    contexts = row["parsed_contexts"]

    context_block = build_context_block(
        contexts,
        max_contexts=3,
        max_chars_per_context=1400
    )

    user_content = f"""Bağlam:
{context_block}

Soru:
{question}

Nihai cevap:"""

    prompt = f"""<s>[INST] {DIRECT_SYSTEM_INSTRUCTION}

{user_content} [/INST]"""

    return prompt


eval_df["direct_answer_rag_prompt"] = eval_df.apply(build_direct_answer_rag_prompt, axis=1)

print(eval_df.loc[0, "direct_answer_rag_prompt"][-2000:])

<s>[INST] Sen Türk hukuku RAG asistanısın.
Yalnızca verilen bağlamlara dayanarak cevap ver.
Kaynakta olmayan bilgiyi uydurma.
Bu değerlendirmede kaynak/citation satırı yazma.
Bağlam numarası veya chunk bilgisini yazma.
Bağlamı kopyalama.
Sadece sorunun kısa ve doğrudan nihai cevabını Türkçe yaz.

Bağlam:
[Bağlam 1]
Cumhurbaşkanlığı seçiminde birinci oylamada gerekli çoğunluğun sağlanamaması halinde 101 inci maddedeki usule göre ikinci oylama yapılır. D. Seçimlerin geriye bırakılması ve ara seçimler [34]

Madde 158 – Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir. [101]

Madde 176 – Anayasanın dayandığı temel görüş ve ilkeleri belirten başlangıç kısmı, Anayasa metnine dahildir.

Soru:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

Nihai cevap: [/INST]


In [16]:
starlar_ft_direct_answers = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    answer = generate_answer_only(
        model=ft_model,
        tokenizer=tokenizer,
        prompt=row["direct_answer_rag_prompt"],
        max_new_tokens=200,
        max_length=MAX_INPUT_LENGTH
    )

    starlar_ft_direct_answers.append(clean_answer(answer))

eval_df["starlar_finetuned_direct_answer"] = starlar_ft_direct_answers

display(eval_df[[
    "question",
    "expected_answer",
    "clean_generated_answer",
    "starlar_finetuned_rag_answer",
    "starlar_finetuned_direct_answer",
    "manual_score"
]])

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [08:02<00:00, 24.12s/it]


,question,expected_answer,clean_generated_answer,starlar_finetuned_rag_answer,starlar_finetuned_direct_answer,manual_score
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",Tartışmalar:\n\n* Cumhurbaşkanlığı seçiminde b...,Tartışmalar: Cumhurbaşkanlığı seçiminde birinc...,0.0
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","[Bağlam 1] Madde 10 – Herkes, dil, ırk, renk, ...","Anayasanın 10. Maddesi ile çelişirse, Anayasan...",0.5
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",Kaynak metindeki açıklama şu kaynak bilgilerin...,Kaynak: TURKISH_LAW_ESKI_LOW_RISK_ONLY - BELGE...,0.5
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 13/5/1981 gün eklendi.,"Geçici madde 20, 13/5/1981 gün ve 2461 sayılı ...","Geçici madde 20, 13/5/1981 gün ve 2461 sayılı ...",0.0
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde hakkının kullanılmasının e...",Kaynak metindeki esas içerik şudur: [Bağlam 1]...,Nihai cevap: Videoda TCK 121 ihlali sabit deği...,0.5
5,Cumhurbaşkanının yemin etme zorunluluğu nedir,"Cumhurbaşkanının yemin etme zorunluluğu, anaya...","Cumhurbaşkanın yemin etme zorunluluğu, Cumhurb...",Cumhurbaşkanının yemin etme zorunluluğu:\n[Bağ...,Cumhurbaşkanının yemin etme zorunluluğu: Cumhu...,0.0
6,Anayasanın 122. Maddesi Nedir?,"Anayasanın 122. maddesi, sıkıyönetim ilanı ve ...",Anayasanın 122. Maddesi yoktur.,Anayasanın 122. Maddesi: e) Anayasanın halkoyl...,Anayasanın halkoylaması sonucu kabulünün ilanı...,0.0
7,Yasama dokunulmazlığının kaldırılmasına karşı ...,Yasama dokunulmazlığının kaldırılmasına veya m...,"Verilen bağlamlara göre, itiraz süresi yedi gü...",Yasama dokunulmazlığının kaldırılmasına ilişki...,"Bu ayrıntılı kaynak, kesin resmî karar metni d...",1.0
8,"Bir siyasi parti, çalışma şartlarının belirli ...","Evet, Anayasanın 50. Maddesi, herkesin çalışma...","Evet, Anayasanın 50. Maddesi, siyasi partileri...","[Bağlam 1] Madde 50 – Kimse, yaşına, cinsiyeti...",Bu açıklama şu kaynak bilgilerine dayandırılma...,0.5
9,"Devlet Denetleme Kurulu, Silahlı Kuvvetler üze...","Anayasanın 108. Maddesi, Silahlı Kuvvetlerin D...","Anayasanın 108. Maddesi, Devlet Denetleme Kuru...",108. Maddesi şu içerik ile karşılaşmaktadır: [...,"Anayasanın 108. Maddesi ile çelişirse de, Devl...",0.0


In [18]:
# Ensure old base automatic metrics exist
eval_df["old_base_token_f1"] = eval_df.apply(
    lambda row: token_f1(row["clean_generated_answer"], row["expected_answer"]),
    axis=1
)

eval_df["old_base_similarity"] = eval_df.apply(
    lambda row: text_similarity(row["clean_generated_answer"], row["expected_answer"]),
    axis=1
)

eval_df["old_base_has_source"] = eval_df["clean_generated_answer"].apply(has_source_citation)


# Ensure Starlar-style fine-tuned automatic metrics exist
eval_df["starlar_ft_token_f1"] = eval_df.apply(
    lambda row: token_f1(row["starlar_finetuned_rag_answer"], row["expected_answer"]),
    axis=1
)

eval_df["starlar_ft_similarity"] = eval_df.apply(
    lambda row: text_similarity(row["starlar_finetuned_rag_answer"], row["expected_answer"]),
    axis=1
)

eval_df["starlar_ft_has_source"] = eval_df["starlar_finetuned_rag_answer"].apply(has_source_citation)


# Direct-answer fine-tuned automatic metrics
eval_df["starlar_direct_token_f1"] = eval_df.apply(
    lambda row: token_f1(row["starlar_finetuned_direct_answer"], row["expected_answer"]),
    axis=1
)

eval_df["starlar_direct_similarity"] = eval_df.apply(
    lambda row: text_similarity(row["starlar_finetuned_direct_answer"], row["expected_answer"]),
    axis=1
)

eval_df["starlar_direct_has_source"] = eval_df["starlar_finetuned_direct_answer"].apply(has_source_citation)


# Valid sample filtering
valid_eval_df = eval_df[eval_df["is_valid_sample"].astype(bool) == True].copy()

three_way_auto_summary_df = pd.DataFrame([
    {
        "method": "Old Best Base Generator",
        "existing_manual_accuracy": valid_eval_df["manual_score"].mean(),
        "mean_token_f1": valid_eval_df["old_base_token_f1"].mean(),
        "mean_text_similarity": valid_eval_df["old_base_similarity"].mean(),
        "source_citation_rate": valid_eval_df["old_base_has_source"].mean(),
        "valid_sample_count": len(valid_eval_df)
    },
    {
        "method": "Starlar Fine-tuned - Starlar-style prompt",
        "existing_manual_accuracy": None,
        "mean_token_f1": valid_eval_df["starlar_ft_token_f1"].mean(),
        "mean_text_similarity": valid_eval_df["starlar_ft_similarity"].mean(),
        "source_citation_rate": valid_eval_df["starlar_ft_has_source"].mean(),
        "valid_sample_count": len(valid_eval_df)
    },
    {
        "method": "Starlar Fine-tuned - Direct-answer prompt",
        "existing_manual_accuracy": None,
        "mean_token_f1": valid_eval_df["starlar_direct_token_f1"].mean(),
        "mean_text_similarity": valid_eval_df["starlar_direct_similarity"].mean(),
        "source_citation_rate": valid_eval_df["starlar_direct_has_source"].mean(),
        "valid_sample_count": len(valid_eval_df)
    }
])

display(three_way_auto_summary_df)

display(eval_df[[
    "question",
    "old_base_token_f1",
    "starlar_ft_token_f1",
    "starlar_direct_token_f1",
    "old_base_similarity",
    "starlar_ft_similarity",
    "starlar_direct_similarity",
    "old_base_has_source",
    "starlar_ft_has_source",
    "starlar_direct_has_source"
]])

,method,existing_manual_accuracy,mean_token_f1,mean_text_similarity,source_citation_rate,valid_sample_count
0,Old Best Base Generator,0.421053,0.286128,0.388491,0.000000,19
1,Starlar Fine-tuned - Starlar-style prompt,NaN,0.144438,0.224722,0.473684,19
2,Starlar Fine-tuned - Direct-answer prompt,NaN,0.156157,0.220360,0.578947,19


,question,old_base_token_f1,starlar_ft_token_f1,starlar_direct_token_f1,old_base_similarity,starlar_ft_similarity,starlar_direct_similarity,old_base_has_source,starlar_ft_has_source,starlar_direct_has_source
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,0.148148,0.100000,0.072727,0.235772,0.106195,0.089552,0,1,1
1,"Bir grup vatandaş, belirli bir etnik grubun di...",0.263158,0.130435,0.131148,0.319392,0.354430,0.218009,0,1,1
2,"Bir grup akademisyen, yaşama hakkının sınırlan...",0.341463,0.109890,0.000000,0.579805,0.269630,0.151815,0,0,1
3,Geçici madde 20 ne zaman eklendi?,0.166667,0.027397,0.033333,0.300000,0.058091,0.074667,0,0,0
4,Videoda TCK 121 ihlali sabit değil mi?,0.000000,0.046512,0.054795,0.210526,0.203226,0.236893,0,0,0
5,Cumhurbaşkanının yemin etme zorunluluğu nedir,0.272727,0.157303,0.186667,0.264249,0.174785,0.205387,0,0,0
6,Anayasanın 122. Maddesi Nedir?,0.428571,0.181818,0.169492,0.481481,0.237288,0.223158,0,1,0
7,Yasama dokunulmazlığının kaldırılmasına karşı ...,0.088235,0.188034,0.125000,0.048507,0.137652,0.027778,0,0,1
8,"Bir siyasi parti, çalışma şartlarının belirli ...",0.341463,0.128205,0.144928,0.476190,0.210896,0.232283,0,0,0
9,"Devlet Denetleme Kurulu, Silahlı Kuvvetler üze...",0.333333,0.150000,0.250000,0.520270,0.200000,0.365805,0,0,1


In [19]:
rag_ft_results_path = f"{metrics_path}/starlar_finetuned_llm_on_old_best_rag_contexts_results_20.csv"
rag_ft_auto_summary_path = f"{metrics_path}/starlar_finetuned_llm_on_old_best_rag_contexts_auto_summary_20.csv"

save_df = eval_df.copy()

save_df["parsed_contexts"] = save_df["parsed_contexts"].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
)

save_df.to_csv(
    rag_ft_results_path,
    index=False,
    encoding="utf-8-sig"
)

three_way_auto_summary_df.to_csv(
    rag_ft_auto_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Updated results saved:", rag_ft_results_path)
print("Updated auto summary saved:", rag_ft_auto_summary_path)

Updated results saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_results_20.csv
Updated auto summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_auto_summary_20.csv


In [20]:
rag_ft_decision_df = pd.DataFrame([
    {
        "experiment": "Starlar fine-tuned LLM on old best RAG contexts",
        "old_best_manual_accuracy": three_way_auto_summary_df.loc[
            three_way_auto_summary_df["method"] == "Old Best Base Generator",
            "existing_manual_accuracy"
        ].values[0],
        "old_best_token_f1": three_way_auto_summary_df.loc[
            three_way_auto_summary_df["method"] == "Old Best Base Generator",
            "mean_token_f1"
        ].values[0],
        "old_best_text_similarity": three_way_auto_summary_df.loc[
            three_way_auto_summary_df["method"] == "Old Best Base Generator",
            "mean_text_similarity"
        ].values[0],
        "starlar_style_token_f1": three_way_auto_summary_df.loc[
            three_way_auto_summary_df["method"] == "Starlar Fine-tuned - Starlar-style prompt",
            "mean_token_f1"
        ].values[0],
        "starlar_style_text_similarity": three_way_auto_summary_df.loc[
            three_way_auto_summary_df["method"] == "Starlar Fine-tuned - Starlar-style prompt",
            "mean_text_similarity"
        ].values[0],
        "direct_prompt_token_f1": three_way_auto_summary_df.loc[
            three_way_auto_summary_df["method"] == "Starlar Fine-tuned - Direct-answer prompt",
            "mean_token_f1"
        ].values[0],
        "direct_prompt_text_similarity": three_way_auto_summary_df.loc[
            three_way_auto_summary_df["method"] == "Starlar Fine-tuned - Direct-answer prompt",
            "mean_text_similarity"
        ].values[0],
        "selected_for_final_rag": False,
        "decision": "Do not replace the old best RAG generator with the Starlar fine-tuned LLM for the original RAG benchmark.",
        "interpretation": "The Starlar fine-tuned LLM strongly improved controlled gold-context generation, but did not improve the old best RAG benchmark when inserted into fixed retrieved contexts. The result is likely limited by retrieval/context mismatch and evaluation format differences."
    }
])

display(rag_ft_decision_df)

rag_ft_decision_path = f"{metrics_path}/starlar_finetuned_llm_on_old_best_rag_contexts_decision_summary.csv"

rag_ft_decision_df.to_csv(
    rag_ft_decision_path,
    index=False,
    encoding="utf-8-sig"
)

print("Decision summary saved:", rag_ft_decision_path)

,experiment,old_best_manual_accuracy,old_best_token_f1,old_best_text_similarity,starlar_style_token_f1,starlar_style_text_similarity,direct_prompt_token_f1,direct_prompt_text_similarity,selected_for_final_rag,decision,interpretation
0,Starlar fine-tuned LLM on old best RAG contexts,0.421053,0.286128,0.388491,0.144438,0.224722,0.156157,0.22036,False,Do not replace the old best RAG generator with...,The Starlar fine-tuned LLM strongly improved c...


Decision summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_decision_summary.csv


In [21]:
eval_summary_df = pd.DataFrame([{
    "experiment": "Starlar fine-tuned LLM on old best RAG contexts",
    "base_model": model_name,
    "fine_tuned_adapter": adapter_path,
    "input_file": old_best_scored_path,
    "eval_sample_size": len(eval_df),
    "valid_sample_count": len(valid_eval_df),
    "retrieval_setup": "Fixed old best retrieved contexts from source-aware + article-aware + Turkish BGE pipeline",
    "old_base_existing_manual_accuracy": valid_eval_df["manual_score"].mean(),
    "old_base_token_f1": three_way_auto_summary_df.loc[
        three_way_auto_summary_df["method"] == "Old Best Base Generator",
        "mean_token_f1"
    ].values[0],
    "starlar_style_token_f1": three_way_auto_summary_df.loc[
        three_way_auto_summary_df["method"] == "Starlar Fine-tuned - Starlar-style prompt",
        "mean_token_f1"
    ].values[0],
    "direct_prompt_token_f1": three_way_auto_summary_df.loc[
        three_way_auto_summary_df["method"] == "Starlar Fine-tuned - Direct-answer prompt",
        "mean_token_f1"
    ].values[0],
    "selected_for_final_rag": False,
    "results_path": rag_ft_results_path,
    "auto_summary_path": rag_ft_auto_summary_path,
    "decision_summary_path": rag_ft_decision_path
}])

eval_summary_path = f"{metrics_path}/starlar_finetuned_llm_on_old_best_rag_contexts_eval_summary_20.csv"

eval_summary_df.to_csv(
    eval_summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(eval_summary_df)

print("Eval summary saved:", eval_summary_path)

,experiment,base_model,fine_tuned_adapter,input_file,eval_sample_size,valid_sample_count,retrieval_setup,old_base_existing_manual_accuracy,old_base_token_f1,starlar_style_token_f1,direct_prompt_token_f1,selected_for_final_rag,results_path,auto_summary_path,decision_summary_path
0,Starlar fine-tuned LLM on old best RAG contexts,mistralai/Mistral-7B-Instruct-v0.2,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...,20,19,Fixed old best retrieved contexts from source-...,0.421053,0.286128,0.144438,0.156157,False,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...


Eval summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_eval_summary_20.csv


In [22]:
files_to_check = [
    rag_ft_results_path,
    rag_ft_auto_summary_path,
    rag_ft_decision_path,
    eval_summary_path
]

print("FINAL CHECK")
print("=" * 80)

for file in files_to_check:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size KB:", round(os.path.getsize(file) / 1024, 2))
    print("-" * 80)

print("Notebook 22 completed.")

FINAL CHECK
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_results_20.csv
Exists: True
Size KB: 130.68
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_auto_summary_20.csv
Exists: True
Size KB: 0.4
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_decision_summary.csv
Exists: True
Size KB: 0.78
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_finetuned_llm_on_old_best_rag_contexts_eval_summary_20.csv
Exists: True
Size KB: 1.1
--------------------------------------------------------------------------------
Notebook 22 completed.
